Gold star schema For Douyin Food & Restaurant Trend BI

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.window import Window

In [0]:
silver_aweme_clean = f"de_e2e.silver.douyin_aweme_clean"
silver_aweme_snapshot = f"de_e2e.silver.aweme_snapshot_delta"
silver_aweme_hashtag = f"de_e2e.silver.douyin_aweme_hashtag"
silver_aweme_media = f"de_e2e.silver.douyin_aweme_media"

bucket = "de-e2e-413612133697-ap-southeast-1-an"
gold_prefix = f"s3://{bucket}/lakehouse/gold/douyin"

dim_date_table = f"de_e2e.gold.dim_date"
dim_creator_table = f"de_e2e.gold.dim_creator"
dim_aweme_table = f"de_e2e.gold.dim_aweme"
dim_hashtag_table = f"de_e2e.gold.dim_hashtag"
dim_media_table = f"de_e2e.gold.dim_media"
fact_aweme_daily_table = f"de_e2e.gold.fact_aweme_daily_performance"
fact_creator_daily_table = f"de_e2e.gold.fact_creator_daily_performance"
fact_hashtag_daily_table = f"de_e2e.gold.fact_hashtag_daily"

paths = {
    dim_date_table: f"{gold_prefix}/dim_date_delta/",
    dim_creator_table: f"{gold_prefix}/dim_creator_delta/",
    dim_aweme_table: f"{gold_prefix}/dim_aweme_delta/",
    dim_hashtag_table: f"{gold_prefix}/dim_hashtag_delta/",
    dim_media_table: f"{gold_prefix}/dim_media_delta/",
    fact_aweme_daily_table: f"{gold_prefix}/fact_aweme_daily_performance_delta/",
    fact_creator_daily_table: f"{gold_prefix}/fact_creator_daily_performance_delta/",
    fact_hashtag_daily_table: f"{gold_prefix}/fact_hashtag_daily_delta/",
}

In [0]:
dbutils.widgets.text("date_start", "2026-02-01", "Start Date (YYYY-MM-DD)")
dbutils.widgets.text("date_end", "2026-06-30", "End Date (YYYY-MM-DD)")

date_start = dbutils.widgets.get("date_start")
date_end = dbutils.widgets.get("date_end")

In [0]:
aweme_clean_df = spark.table(silver_aweme_clean)
aweme_snapshot_df = spark.table(silver_aweme_snapshot)
hashtag_df = spark.table(silver_aweme_hashtag)
media_df = spark.table(silver_aweme_media)

print(f"aweme_clean rows    = {aweme_clean_df.count()}")
print(f"aweme_snapshot rows = {aweme_snapshot_df.count()}")
print(f"hashtag rows        = {hashtag_df.count()}")
print(f"media rows          = {media_df.count()}")

In [0]:
def write_delta_replace(df: DataFrame, table_name: str, table_path: str) -> None:
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(table_path)
    )
    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS {table_name}
        USING DELTA
        LOCATION '{table_path}'
        """
    )


def key_sha2(*cols: str):
    return F.sha2(F.concat_ws("||", *[F.coalesce(F.col(col).cast("string"), F.lit("")) for col in cols]), 256)


In [0]:
dim_date_df = (
    spark.sql(
        f"""
        SELECT explode(sequence(to_date('{date_start}'), to_date('{date_end}'), interval 1 day)) AS date
        """
    )
    .select(
        F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
        F.col("date"),
        F.year("date").alias("year"),
        F.quarter("date").alias("quarter"),
        F.month("date").alias("month"),
        F.dayofmonth("date").alias("day"),
        F.date_format("date", "E").alias("day_of_week"),
        F.weekofyear("date").alias("week_of_year"),
    )
)

write_delta_replace(dim_date_df, dim_date_table, paths[dim_date_table])

In [0]:
creator_window = Window.partitionBy("account_id", "author_uid").orderBy(
    F.col("landing_generated_at").desc_nulls_last(),
    F.col("bronze_ingested_at").desc_nulls_last(),
)

dim_creator_df = (
    aweme_clean_df
    .where(F.col("author_uid").isNotNull())
    .withColumn("rn", F.row_number().over(creator_window))
    .withColumn("creator_key", key_sha2("account_id", "author_uid"))
    .withColumn("first_seen_at", F.min("landing_generated_at").over(Window.partitionBy("account_id", "author_uid")))
    .withColumn("last_seen_at", F.max("landing_generated_at").over(Window.partitionBy("account_id", "author_uid")))
    .where(F.col("rn") == 1)
    .select(
        "creator_key",
        "account_id",
        "author_uid",
        "author_sec_uid",
        "author_nickname",
        "author_signature",
        "author_custom_verify",
        "author_enterprise_verify_reason",
        "first_seen_at",
        "last_seen_at",
    )
)

write_delta_replace(dim_creator_df, dim_creator_table, paths[dim_creator_table])


In [0]:
dim_aweme_df = (
    aweme_clean_df
    .where(F.col("aweme_id").isNotNull())
    .withColumn("aweme_key", key_sha2("account_id", "aweme_id"))
    .withColumn("creator_key", key_sha2("account_id", "author_uid"))
    .withColumn("created_date_key", F.date_format(F.to_date("created_at"), "yyyyMMdd").cast("int"))
    .select(
        "aweme_key",
        "account_id",
        "aweme_id",
        "creator_key",
        "description",
        "item_title",
        "share_url",
        "created_at",
        "created_date_key",
        "duration_ms",
        "duration_seconds",
        "media_type",
        "aweme_type",
        "region",
        "is_ads",
        "is_top",
        "prevent_download",
    )
)

write_delta_replace(dim_aweme_df, dim_aweme_table, paths[dim_aweme_table])


In [0]:
hashtag_window = Window.partitionBy("hashtag_name").orderBy(
    F.col("landing_generated_at").desc_nulls_last(),
    F.col("bronze_ingested_at").desc_nulls_last(),
)

dim_hashtag_df = (
    hashtag_df
    .where(F.col("hashtag_name").isNotNull())
    .withColumn("hashtag_name_clean", F.lower(F.trim(F.col("hashtag_name"))))
    .withColumn("rn", F.row_number().over(hashtag_window))
    .withColumn("hashtag_key", F.sha2(F.col("hashtag_name_clean"), 256))
    .withColumn("first_seen_at", F.min("landing_generated_at").over(Window.partitionBy("hashtag_name")))
    .withColumn("last_seen_at", F.max("landing_generated_at").over(Window.partitionBy("hashtag_name")))
    .where(F.col("rn") == 1)
    .select(
        "hashtag_key",
        "hashtag_name",
        "hashtag_id",
        "hashtag_type",
        "is_commerce",
        "first_seen_at",
        "last_seen_at",
    )
)

write_delta_replace(dim_hashtag_df, dim_hashtag_table, paths[dim_hashtag_table])


In [0]:
media_window = Window.partitionBy("account_id", "aweme_id", "media_type", "media_index").orderBy(
    F.col("manifest_generated_at").desc_nulls_last(),
    F.col("bronze_ingested_at").desc_nulls_last(),
    F.col("source_file").desc_nulls_last(),
)

dim_media_df = (
    media_df
    .where(F.col("aweme_id").isNotNull())
    .where(F.col("media_type").isNotNull())
    .where(F.col("media_index").isNotNull())
    .withColumn("rn", F.row_number().over(media_window))
    .where(F.col("rn") == 1)
    .withColumn("media_key", key_sha2("account_id", "aweme_id", "media_type", "media_index"))
    .withColumn("aweme_key", key_sha2("account_id", "aweme_id"))
    .select(
        "media_key",
        "aweme_key",
        "account_id",
        "aweme_id",
        "media_type",
        "media_index",
        "s3_key",
        "s3_url",
        "bytes",
        "content_type",
        "status",
    )
)

write_delta_replace(dim_media_df, dim_media_table, paths[dim_media_table])


In [0]:
latest_daily_window = Window.partitionBy("account_id", "aweme_id", "snapshot_date").orderBy(
    F.col("landing_generated_at").desc_nulls_last(),
    F.col("bronze_ingested_at").desc_nulls_last(),
    F.col("source_file").desc_nulls_last(),
)

base_daily_df = (
    aweme_snapshot_df
    .where(F.col("aweme_id").isNotNull())
    .where(F.col("snapshot_date").isNotNull())
    .withColumn("rn", F.row_number().over(latest_daily_window))
    .where(F.col("rn") == 1)
    .drop("rn")
    .withColumn("aweme_key", key_sha2("account_id", "aweme_id"))
    .withColumn("creator_key", key_sha2("account_id", "author_uid"))
    .withColumn("snapshot_date_key", F.date_format("snapshot_date", "yyyyMMdd").cast("int"))
    .withColumn("like_count", F.coalesce(F.col("like_count"), F.lit(0)))
    .withColumn("comment_count", F.coalesce(F.col("comment_count"), F.lit(0)))
    .withColumn("share_count", F.coalesce(F.col("share_count"), F.lit(0)))
    .withColumn("collect_count", F.coalesce(F.col("collect_count"), F.lit(0)))
    .withColumn("recommend_count", F.coalesce(F.col("recommend_count"), F.lit(0)))
    .withColumn("play_count", F.coalesce(F.col("play_count"), F.lit(0)))
    .withColumn("admire_count", F.coalesce(F.col("admire_count"), F.lit(0)))
    .withColumn(
        "engagement_score",
        F.col("like_count")
        + F.col("comment_count") * F.lit(2)
        + F.col("share_count") * F.lit(3)
        + F.col("collect_count") * F.lit(2)
        + F.col("recommend_count"),
    )
    .withColumn(
        "engagement_rate_by_play",
        F.when(
            F.col("play_count") > 0,
            (F.col("like_count") + F.col("comment_count") + F.col("share_count") + F.col("collect_count")) / F.col("play_count"),
        ),
    )
)

history_window = Window.partitionBy("account_id", "aweme_id").orderBy("snapshot_date")

fact_aweme_daily_df = (
    base_daily_df
    .withColumn("like_delta", F.col("like_count") - F.lag("like_count").over(history_window))
    .withColumn("comment_delta", F.col("comment_count") - F.lag("comment_count").over(history_window))
    .withColumn("share_delta", F.col("share_count") - F.lag("share_count").over(history_window))
    .withColumn("collect_delta", F.col("collect_count") - F.lag("collect_count").over(history_window))
    .withColumn("engagement_delta", F.col("engagement_score") - F.lag("engagement_score").over(history_window))
    .select(
        "aweme_key",
        "creator_key",
        "snapshot_date_key",
        "account_id",
        "aweme_id",
        "source_file",
        "landing_generated_at",
        "like_count",
        "comment_count",
        "share_count",
        "collect_count",
        "recommend_count",
        "play_count",
        "admire_count",
        "engagement_score",
        "engagement_rate_by_play",
        "like_delta",
        "comment_delta",
        "share_delta",
        "collect_delta",
        "engagement_delta",
    )
)

write_delta_replace(fact_aweme_daily_df, fact_aweme_daily_table, paths[fact_aweme_daily_table])


In [0]:
creator_daily_base_df = spark.table(fact_aweme_daily_table)

top_aweme_window = Window.partitionBy("creator_key", "snapshot_date_key").orderBy(F.col("engagement_score").desc_nulls_last())

top_aweme_df = (
    creator_daily_base_df
    .withColumn("rn", F.row_number().over(top_aweme_window))
    .where(F.col("rn") == 1)
    .select("creator_key", "snapshot_date_key", F.col("aweme_key").alias("top_aweme_key"))
)

fact_creator_daily_df = (
    creator_daily_base_df
    .groupBy("creator_key", "snapshot_date_key")
    .agg(
        F.countDistinct("aweme_key").alias("aweme_count"),
        F.sum("like_count").alias("total_like_count"),
        F.sum("comment_count").alias("total_comment_count"),
        F.sum("share_count").alias("total_share_count"),
        F.sum("collect_count").alias("total_collect_count"),
        F.sum("engagement_score").alias("total_engagement_score"),
        F.avg("engagement_score").alias("avg_engagement_score"),
    )
    .join(top_aweme_df, on=["creator_key", "snapshot_date_key"], how="left")
)

write_delta_replace(fact_creator_daily_df, fact_creator_daily_table, paths[fact_creator_daily_table])


In [0]:
fact_aweme_daily_df = spark.table(fact_aweme_daily_table)

hashtag_mapping_df = (
    hashtag_df
    .where(F.col("hashtag_name").isNotNull())
    .withColumn("hashtag_key", F.sha2(F.lower(F.trim(F.col("hashtag_name"))), 256))
    .withColumn("aweme_key", key_sha2("account_id", "aweme_id"))
    .select("hashtag_key", "aweme_key")
    .dropDuplicates()
)

fact_hashtag_daily_df = (
    hashtag_mapping_df
    .join(fact_aweme_daily_df, on="aweme_key", how="inner")
    .groupBy("hashtag_key", "snapshot_date_key")
    .agg(
        F.countDistinct("aweme_key").alias("aweme_count"),
        F.countDistinct("creator_key").alias("creator_count"),
        F.sum("like_count").alias("total_like_count"),
        F.sum("comment_count").alias("total_comment_count"),
        F.sum("share_count").alias("total_share_count"),
        F.sum("collect_count").alias("total_collect_count"),
        F.sum("engagement_score").alias("total_engagement_score"),
        F.avg("engagement_score").alias("avg_engagement_score"),
    )
)

write_delta_replace(fact_hashtag_daily_df, fact_hashtag_daily_table, paths[fact_hashtag_daily_table])


In [0]:
for table_name in [
    dim_date_table,
    dim_creator_table,
    dim_aweme_table,
    dim_hashtag_table,
    dim_media_table,
    fact_aweme_daily_table,
    fact_creator_daily_table,
    fact_hashtag_daily_table,
]:
    row_count = spark.table(table_name).count()
    print(f"{table_name}: {row_count} rows")